In [0]:
# ════════════════════════════════════════════════════════════════
# MapleBank Shared Utilities — PII Masking & Validation
# ONE SOURCE OF TRUTH for compliance-critical logic (PIPEDA/OSFI)
# Usage in any notebook:  %run ../utils/masking_functions
# ════════════════════════════════════════════════════════════════
import pyspark.sql.functions as F

def mask_sin(col_name="sin"):
    """PIPEDA-compliant SIN mask: show last 3 digits only.
    872-948-301  →  ***-***-301"""
    return F.when(
        F.col(col_name).isNotNull(),
        F.concat(F.lit("***-***-"), F.substring(F.col(col_name), 8, 3))
    ).otherwise(F.lit("***-***-***"))

def mask_account(col_name="account_number"):
    """Account number mask: show last 3 digits.
    4859771  →  ****771"""
    return F.when(
        F.col(col_name).isNotNull(),
        F.concat(F.lit("****"), F.substring(F.col(col_name), -3, 3))
    ).otherwise(F.lit("****"))

def postal_to_fsa(col_name="postal_code"):
    """Reduce Canadian postal code to Forward Sortation Area (first 3 chars).
    M5V 2T6  →  M5V   (FSA ≈ 7,000 people — safe for analytics)"""
    return F.when(
        F.col(col_name).isNotNull(),
        F.upper(F.substring(F.col(col_name), 1, 3))
    ).otherwise(F.lit("UNK"))

def mask_email(col_name="email"):
    """Keep first char + domain:  john.smith@example.ca → j***@example.ca"""
    return F.when(
        F.col(col_name).contains("@"),
        F.concat(
            F.substring(F.col(col_name), 1, 1),
            F.lit("***@"),
            F.split(F.col(col_name), "@").getItem(1)
        )
    ).otherwise(F.lit("***"))

def validate_transit(col_name="transit_number"):
    """Canadian transit numbers are exactly 5 digits."""
    return F.col(col_name).rlike("^[0-9]{5}$")

def validate_province(col_name="province"):
    """Valid Canadian province/territory codes."""
    return F.col(col_name).isin(
        "ON","QC","BC","AB","MB","SK","NS","NB","NL","PE","YT","NT","NU"
    )

def add_audit_columns(df, source_name, business_date):
    """Bronze-layer metadata: source traceability for OSFI audit."""
    return df \
        .withColumn("_source_file",         F.lit(source_name)) \
        .withColumn("_ingestion_timestamp", F.current_timestamp()) \
        .withColumn("_business_date",       F.lit(business_date))

print("✔ MapleBank utils loaded: mask_sin, mask_account, postal_to_fsa, "
      "mask_email, validate_transit, validate_province, add_audit_columns")

✔ MapleBank utils loaded: mask_sin, mask_account, postal_to_fsa, mask_email, validate_transit, validate_province, add_audit_columns
